# 7. Autoresearch-agent memory-debug loop

This is the case study requested by the proposal: an autonomous research agent generates ideas, retrieves prior experiment memories, and is debugged.

## Proposal Part 4 case study mapping

The proposal's Part 4 case study describes single-GPU/autonomous research with:

- 100+ overnight experiments.
- Working memory over recent experiments.
- Episodic compressed memory.
- Semantic/LLM-extracted cached memory.
- Fewer redundant experiments, faster convergence, and reduced API cost.

This notebook implements the tutorial-scale analogue: real benchmark outcomes become experiment memories, the agent proposes new ideas, retrieves relevant prior results, then decides **keep**, **revise**, or **discard** while diagnosing memory failures.

In [ ]:
import os
# Optional OpenAI-compatible idea generation. The default executed path remains offline.
# To use the API, set OPENAI_API_KEY outside the notebook and run experiment/run.py with:
# --backend openai-compatible --base-url http://127.0.0.1:8317/api/provider/codex/v1 --model gpt-5.5
print("OPENAI_API_KEY present:", bool(os.environ.get("OPENAI_API_KEY")))

In [ ]:
from pathlib import Path
import json
import os
import sys

# Colab guidance:
# 1. Upload or clone this repository into the Colab runtime.
# 2. Set PROJECT_ROOT to the repository root if auto-detection fails.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "experiment" / "run.py").exists():
    candidates = [Path("/content/kdd"), Path("/content/drive/MyDrive/kdd"), Path("/Users/syaikhipin/Documents/Phd/Publish Paper/kdd")]
    PROJECT_ROOT = next((p for p in candidates if (p / "experiment" / "run.py").exists()), PROJECT_ROOT)

EXPERIMENT_DIR = PROJECT_ROOT / "experiment"
RESULTS_DIR = EXPERIMENT_DIR / "results"
sys.path.insert(0, str(EXPERIMENT_DIR))
print("PROJECT_ROOT =", PROJECT_ROOT)
print("Experiment code exists:", (EXPERIMENT_DIR / "run.py").exists())

In [ ]:
import subprocess
latest_real_metrics = sorted(RESULTS_DIR.glob("run_*_real_metrics.json"))[-1]
cmd = [
    sys.executable, str(EXPERIMENT_DIR / "run.py"),
    "--mode", "autoresearch-agent",
    "--backend", "offline",
    "--benchmark-metrics", str(latest_real_metrics),
    "--use-cases", "locomo", "autoresearch", "hpo", "memoryarena", "longmemeval", "lcbench",
    "--ideas-per-case", "2",
    "--top-k", "5",
    "--eval-backend", "offline",
    "--eval-limit", "12",
]
print(" ".join(map(str, cmd)))
result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True, timeout=120)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0

## Semantic evaluation of generated ideas

Notebook 7 now evaluates generated idea records with the offline semantic evaluator. Rhesis and Semantica can use the same `--eval-backend` flag for small maintainer smoke tests outside the live tutorial path.

In [ ]:
latest = sorted(RESULTS_DIR.glob("run_*_autoresearch_agent_raw.json"))[-1]
print("Reading", latest)
trace = json.loads(latest.read_text())
for r in trace["records"]:
    print(r["step"], r["idea_id"], r["dataset"], r["decision"], r["failure_category"], "novelty=", r["novelty_score"], "signal=", r["observed_signal"])

Guidance: use this for the Part 4 case study. Discuss why the agent keeps some ideas, revises unsupported ones, and discards redundant ideas.